<a href="https://colab.research.google.com/github/wondrv/py/blob/main/cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# 1. MEMBUAT DATASET (MINIMAL 20 DATA)
# Label: 1 untuk Hoaks, 0 untuk Asli
data = {
    'teks': [
        'Pemerintah akan membagikan kuota internet gratis 100GB untuk semua operator', # Hoaks
        'Bank Indonesia merilis uang baru pecahan 1 juta rupiah', # Hoaks
        'Telah ditemukan vaksin COVID-19 dalam bentuk permen jahe', # Hoaks
        'Pesan berantai ini akan membawa sial jika tidak disebar', # Hoaks
        'Unicorn ditemukan hidup di pedalaman hutan Kalimantan', # Hoaks
        'Rendang resmi dinobatkan sebagai makanan terenak di dunia oleh CNN', # Asli
        'BUMN membuka lebih dari 2.700 lowongan kerja dalam rekrutmen bersama', # Asli
        'Kemenkes konfirmasi adanya penurunan kasus stunting di Indonesia', # Asli
        'Pertamina sesuaikan harga BBM non-subsidi per 1 Juli 2025', # Asli
        'Presiden Jokowi resmikan bendungan baru di Jawa Tengah', # Asli
        'Minum air kelapa bisa langsung membersihkan paru-paru perokok', # Hoaks
        'Ada razia STNK besar-besaran serentak di seluruh Indonesia besok', # Hoaks
        'NASA mengumumkan bumi akan gelap selama 3 hari', # Hoaks
        'BMKG mengeluarkan peringatan dini cuaca ekstrem untuk beberapa wilayah', # Asli
        'Polri berhasil menangkap pelaku penyebar berita bohong terkait pemilu', # Asli
        'Telur ayam dari China mengandung plastik berbahaya', # Hoaks
        'Gerhana matahari total akan terlihat jelas di seluruh nusantara', # Hoaks (spesifik lokasi)
        'Timnas Indonesia berhasil lolos ke putaran ketiga kualifikasi piala dunia', # Asli
        'BI melaporkan inflasi bulan Juni 2025 berada di level 3%', # Asli
        'Gubernur DKI Jakarta mengumumkan kebijakan baru terkait transportasi publik', # Asli
    ],
    'label': [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0]
}
df = pd.DataFrame(data)

print("Contoh Data:")
print(df.head())
print("\nJumlah Data:", len(df))

# 2. PRA-PEMROSESAN TEKS
max_words = 1000  # Ukuran kosakata
max_len = 50      # Panjang maksimal setiap sekuens/berita

tokenizer = Tokenizer(num_words=max_words, oov_token="<unk>")
tokenizer.fit_on_texts(df['teks'])
sequences = tokenizer.texts_to_sequences(df['teks'])
X = pad_sequences(sequences, maxlen=max_len)
y = np.array(df['label'])

# Split data menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nBentuk data training (X_train):", X_train.shape)
print("Bentuk data testing (X_test):", X_test.shape)

# 3. MEMBUAT MODEL CNN
embedding_dim = 50 # Dimensi vektor untuk setiap kata

model = Sequential([
    # Layer 1: Embedding - Mengubah input angka menjadi vektor padat
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),

    # Layer 2: Conv1D - Mengekstrak fitur dengan 128 filter, kernel size 5
    Conv1D(filters=128, kernel_size=5, activation='relu'),

    # Layer 3: Pooling - Mengambil fitur terpenting
    GlobalMaxPooling1D(),

    # Layer 4: Fully Connected - Lapisan pemrosesan sebelum output
    Dense(64, activation='relu'),

    # Layer 5: Output - Menghasilkan probabilitas (0 atau 1)
    Dense(1, activation='sigmoid')
])

# Kompilasi model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

# 4. TRAINING MODEL
print("\n--- Memulai Training ---")
history = model.fit(X_train, y_train,
                    epochs=20,
                    batch_size=8,
                    validation_data=(X_test, y_test),
                    verbose=1)
print("--- Training Selesai ---")

# 5. EVALUASI (VALIDASI/TESTING)
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nAkurasi pada data testing: {accuracy*100:.2f}%")

# 6. PENGUJIAN PADA BERITA BARU
berita_baru = [
    "Selamat anda mendapatkan hadiah undian sebesar 100 juta dari bank resmi", # Prediksi: Hoaks
    "Indeks harga saham gabungan ditutup menguat pada perdagangan hari ini" # Prediksi: Asli
]

# Proses berita baru sama seperti data training
seq_baru = tokenizer.texts_to_sequences(berita_baru)
pad_baru = pad_sequences(seq_baru, maxlen=max_len)

# Lakukan prediksi
prediksi = model.predict(pad_baru)

print("\n--- Hasil Prediksi Berita Baru ---")
for i, teks in enumerate(berita_baru):
    label = "Hoaks" if prediksi[i] > 0.5 else "Asli"
    print(f"Teks: '{teks}'\nPrediksi: {label} (Score: {prediksi[i][0]:.4f})\n")

Contoh Data:
                                                teks  label
0  Pemerintah akan membagikan kuota internet grat...      1
1  Bank Indonesia merilis uang baru pecahan 1 jut...      1
2  Telah ditemukan vaksin COVID-19 dalam bentuk p...      1
3  Pesan berantai ini akan membawa sial jika tida...      1
4  Unicorn ditemukan hidup di pedalaman hutan Kal...      1

Jumlah Data: 20

Bentuk data training (X_train): (16, 50)
Bentuk data testing (X_test): (4, 50)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


--- Memulai Training ---
Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 483ms/step - accuracy: 0.2917 - loss: 0.6948 - val_accuracy: 0.2500 - val_loss: 0.6969
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.8333 - loss: 0.6738 - val_accuracy: 0.2500 - val_loss: 0.7106
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - accuracy: 0.5833 - loss: 0.6536 - val_accuracy: 0.2500 - val_loss: 0.7219
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.5833 - loss: 0.6361 - val_accuracy: 0.2500 - val_loss: 0.7308
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.7083 - loss: 0.6145 - val_accuracy: 0.2500 - val_loss: 0.7392
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.6667 - loss: 0.6002 - val_accuracy: 0.2500 - val_loss: 0.7446
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 1.0000 - loss: 0.5754 - val_accuracy: 0.2500 - val_loss: 0.7509
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 1.0000 - loss: 0.5449 - val_accurac